# 01 — Write the Finding

Analysis scaffold for the lerobot-bench writeup. Loads
`results/sweep.parquet` if present, falls back to a deterministic
synthetic parquet (3 policies × 2 envs × 5 seeds × 50 episodes) so
this notebook executes end-to-end *before* the full sweep produces real
data.

The same notebook runs against the real sweep results post-Day-5 with
no edits — only the `RESULTS_PARQUET` constant in cell 1 changes.

Pipeline:

1. Imports + load (with synthetic fallback).
2. Schema + sanity (every cell has the contracted episode count).
3. Leaderboard table (Wilson 95% CI half-widths via `stats.wilson_ci`).
4. Bootstrap CIs by `(policy, env, seed)` cell — forest plot.
5. Paired comparisons — `paired_delta_bootstrap` + `paired_wilcoxon` +
   `cohens_h` for the two policies sharing the most envs.
6. Failure-mode preview — placeholder bar chart, swaps in real
   `docs/FAILURE_TAXONOMY.md` labels on Day 7.
7. The finding paragraph (markdown, hedged on synthetic).
8. Reproducibility footer.

> **Stats-rigor contract.** Every numeric claim in this notebook cites
> the function in `lerobot_bench.stats` that produced it, states N, and
> for bootstrap intervals records `n_resamples=10000` + `random_state=42`.
> Bullet-checked at the bottom of cell 5.


In [ ]:
# Cell 1 - Imports + load (with synthetic fallback)
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make `lerobot_bench` importable from the repo root regardless of
# where the kernel was launched. Jumps two levels up from `notebooks/`.
REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / "src" / "lerobot_bench").is_dir():
    pass  # launched from repo root, default
elif (REPO_ROOT.parent / "src" / "lerobot_bench").is_dir():
    REPO_ROOT = REPO_ROOT.parent
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import lerobot_bench
from lerobot_bench.checkpointing import RESULT_SCHEMA
from lerobot_bench.stats import (
    bootstrap_ci,
    cohens_h,
    paired_delta_bootstrap,
    paired_wilcoxon,
    wilson_ci,
)

print("lerobot_bench version:", lerobot_bench.__version__)
print("RESULT_SCHEMA columns:", RESULT_SCHEMA)

# --- locate parquet -------------------------------------------------- #
RESULTS_PARQUET = REPO_ROOT / "results" / "sweep.parquet"
USING_SYNTHETIC = not RESULTS_PARQUET.exists()

if USING_SYNTHETIC:
    print(
        f"\n{RESULTS_PARQUET} does not exist — generating synthetic parquet for scaffold execution."
    )
else:
    print(f"\nLoading real sweep results from {RESULTS_PARQUET}.")

In [ ]:
# Cell 1b - Synthetic fallback generator
# Deterministic 3 policies x 2 envs x 5 seeds x 50 episodes parquet.
# Per-policy success rates picked so cell 5's paired comparison is
# non-trivial (DiffPolicy beats ACT on PushT; tie on Aloha) but the
# random baseline is clearly worst on both.


def _synthetic_results_df(seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    policies = ["diffusion_policy", "act", "random"]
    envs = ["pusht", "aloha_transfer_cube"]
    n_seeds = 5
    n_episodes = 50

    # Per-(policy, env) Bernoulli mean for synthetic outcomes.
    p_table = {
        ("diffusion_policy", "pusht"): 0.78,
        ("diffusion_policy", "aloha_transfer_cube"): 0.62,
        ("act", "pusht"): 0.66,
        ("act", "aloha_transfer_cube"): 0.61,
        ("random", "pusht"): 0.04,
        ("random", "aloha_transfer_cube"): 0.00,
    }

    rows = []
    for policy in policies:
        for env in envs:
            p = p_table[(policy, env)]
            for seed in range(n_seeds):
                for ep in range(n_episodes):
                    success = bool(rng.random() < p)
                    ret = float(rng.uniform(0.0, 1.0)) if success else float(rng.uniform(0.0, 0.6))
                    rows.append(
                        {
                            "policy": policy,
                            "env": env,
                            "seed": seed,
                            "episode_index": ep,
                            "success": success,
                            "return_": ret,
                            "n_steps": int(rng.integers(50, 400)),
                            "wallclock_s": float(rng.uniform(2.0, 30.0)),
                            "video_sha256": "synthetic",
                            "code_sha": "synthetic",
                            "lerobot_version": "0.5.1",
                            "timestamp_utc": "2026-05-03T00:00:00Z",
                        }
                    )
    return pd.DataFrame(rows, columns=list(RESULT_SCHEMA))


df = _synthetic_results_df(seed=0) if USING_SYNTHETIC else pd.read_parquet(RESULTS_PARQUET)

print(f"Loaded {len(df):,} rows.")
df.head()

In [ ]:
# Cell 2 - Schema + sanity
# Every claim downstream depends on the parquet conforming to
# RESULT_SCHEMA. This cell asserts the column set, episode count per
# (policy, env, seed) cell, and absence of nulls in critical columns.

assert set(df.columns) == set(
    RESULT_SCHEMA
), f"Schema drift: got {set(df.columns)} expected {set(RESULT_SCHEMA)}"

per_cell = (
    df.groupby(["policy", "env", "seed"])["episode_index"]
    .agg(["count", "min", "max", "nunique"])
    .reset_index()
)
per_cell.columns = ["policy", "env", "seed", "n_rows", "ep_min", "ep_max", "ep_unique"]

# Contract per docs/DESIGN.md - 5 seeds x 50 episodes per cell. The
# auto-downscope rule may reduce per-cell episode count for slow VLAs
# in the real sweep; flag any cell that is not the contracted 50 so the
# methods section can mention it explicitly.
contracted_episodes = 50
non_contract = per_cell[per_cell["n_rows"] != contracted_episodes]
if len(non_contract) > 0:
    print(
        f"WARNING: {len(non_contract)} cells deviate from the {contracted_episodes}-episode contract:"
    )
    print(non_contract.to_string(index=False))
else:
    print(f"OK: all {len(per_cell)} cells have exactly {contracted_episodes} episodes.")

# Null guard on critical columns - success / return_ / steps must not
# be null even for crashed episodes (the eval orchestrator records
# success=False / return_=0.0 on exception).
critical = ["policy", "env", "seed", "episode_index", "success"]
nulls = df[critical].isnull().sum()
if nulls.sum() > 0:
    print("ERROR: nulls in critical columns:")
    print(nulls[nulls > 0])
else:
    print("OK: no nulls in critical columns.")

per_cell.head(10)

In [ ]:
# Cell 3 - Leaderboard table
# One row per (policy, env) cell, success rate + Wilson 95% CI
# half-width. Mirrors space._helpers.compute_leaderboard_table so the
# numbers in this notebook agree with the public Space numbers row by
# row.


def leaderboard_table(df_: pd.DataFrame, ci: float = 0.95) -> pd.DataFrame:
    rows = []
    for (policy, env), cell in df_.groupby(["policy", "env"], sort=False):
        n = len(cell)
        successes = int(cell["success"].sum())
        rate = successes / n
        lo, hi = wilson_ci(successes, n, ci=ci)
        rows.append(
            {
                "policy": policy,
                "env": env,
                "n_episodes": n,
                "n_successes": successes,
                "success_rate": rate,
                "wilson_ci_low": lo,
                "wilson_ci_high": hi,
                "wilson_half_width": (hi - lo) / 2.0,
            }
        )
    out = pd.DataFrame(rows)
    out = out.sort_values(
        ["success_rate", "policy", "env"], ascending=[False, True, True], ignore_index=True
    )
    return out


leaderboard = leaderboard_table(df)
print(f"Leaderboard ({'SYNTHETIC' if USING_SYNTHETIC else 'REAL'} data):")
leaderboard

In [ ]:
# Cell 4 - Bootstrap CIs by cell + forest plot
# Per-(policy, env, seed) cell: bootstrap_ci over the 50 episode
# outcomes. Plotted as a forest (one row per cell, grouped visually
# by policy). Bootstrap settings locked: n_resamples=10000,
# random_state=42 - both surface in the figure caption so a stats-rigor
# reviewer can re-derive any interval from the parquet.

N_RESAMPLES = 10_000
RANDOM_STATE = 42
CI_LEVEL = 0.95

forest_rows = []
for (policy, env, seed), cell in df.groupby(["policy", "env", "seed"]):
    outcomes = cell["success"].to_numpy().astype(bool)
    rng = np.random.default_rng(RANDOM_STATE)
    res = bootstrap_ci(outcomes, n_resamples=N_RESAMPLES, ci=CI_LEVEL, rng=rng)
    forest_rows.append(
        {
            "policy": policy,
            "env": env,
            "seed": seed,
            "n": len(outcomes),
            "mean": res.mean,
            "ci_low": res.lo,
            "ci_high": res.hi,
        }
    )
forest = pd.DataFrame(forest_rows).sort_values(["policy", "env", "seed"], ignore_index=True)

# --- forest plot ----------------------------------------------------- #
fig, ax = plt.subplots(figsize=(7, 0.35 * len(forest) + 1.2))
labels = [f"{r.policy} | {r.env} | seed {r.seed}" for r in forest.itertuples()]
y = np.arange(len(forest))
ax.errorbar(
    forest["mean"],
    y,
    xerr=[forest["mean"] - forest["ci_low"], forest["ci_high"] - forest["mean"]],
    fmt="o",
    color="steelblue",
    ecolor="steelblue",
    capsize=3,
    markersize=4,
)
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis()
ax.set_xlim(-0.05, 1.05)
ax.set_xlabel("success rate (95% bootstrap CI)")
ax.set_title(
    f"Per-cell success rates - {'SYNTHETIC' if USING_SYNTHETIC else 'REAL'} - "
    f"bootstrap n_resamples={N_RESAMPLES:,}, seed={RANDOM_STATE}"
)
ax.axvline(0.5, color="grey", linestyle="--", linewidth=0.5)
fig.tight_layout()
plt.show()
forest.head()

In [ ]:
# Cell 5 - Paired comparisons
# For the two policies that share the most envs, run paired
# Delta-bootstrap AND paired Wilcoxon per env. cohens_h gives the
# proportion-difference effect size. Pairing is on (seed, episode) so
# both arrays must have identical (seed, episode_index) rows in the
# same order.


def shared_env_policy_pair(df_: pd.DataFrame) -> tuple[str, str] | None:
    """Pick the two policies with the largest shared env set."""
    by_policy = df_.groupby("policy")["env"].apply(lambda s: frozenset(s.unique()))
    policies = list(by_policy.index)
    best = None
    best_n = -1
    for i in range(len(policies)):
        for j in range(i + 1, len(policies)):
            shared = by_policy[policies[i]] & by_policy[policies[j]]
            if len(shared) > best_n:
                best_n = len(shared)
                best = (policies[i], policies[j])
    return best


pair = shared_env_policy_pair(df)
assert pair is not None, "no policy pair found"
p_a, p_b = pair
print(f"Paired comparison: {p_a!r} vs {p_b!r}")

paired_results = []
for env in sorted(df["env"].unique()):
    a = df[(df["policy"] == p_a) & (df["env"] == env)].sort_values(["seed", "episode_index"])
    b = df[(df["policy"] == p_b) & (df["env"] == env)].sort_values(["seed", "episode_index"])
    if len(a) == 0 or len(b) == 0 or len(a) != len(b):
        print(f"  skip env={env!r}: incomplete pair (a={len(a)}, b={len(b)})")
        continue
    a_out = a["success"].to_numpy().astype(bool)
    b_out = b["success"].to_numpy().astype(bool)
    rng = np.random.default_rng(RANDOM_STATE)
    delta = paired_delta_bootstrap(a_out, b_out, n_resamples=N_RESAMPLES, ci=CI_LEVEL, rng=rng)
    wilcox = paired_wilcoxon(a_out, b_out)
    h = cohens_h(float(a_out.mean()), float(b_out.mean()))

    # MDE / power flag - Wilson half-width at p=0.5, N=250 is ~0.062;
    # therefore deltas with |delta| < 2 * 0.062 = 0.124 are inconclusive
    # at this N regardless of p-value (per docs/DESIGN.md MDE note,
    # restated in paper Methods).
    n = len(a_out)
    wh = (wilson_ci(n // 2, n, ci=CI_LEVEL)[1] - wilson_ci(n // 2, n, ci=CI_LEVEL)[0]) / 2.0
    inconclusive = abs(delta.mean) < 2 * wh

    paired_results.append(
        {
            "env": env,
            "n_pairs": n,
            "rate_a": float(a_out.mean()),
            "rate_b": float(b_out.mean()),
            "delta_mean": delta.mean,
            "delta_ci_low": delta.lo,
            "delta_ci_high": delta.hi,
            "wilcoxon_p": wilcox.pvalue,
            "wilcoxon_n_pairs": wilcox.n_pairs,
            "cohens_h": h,
            "wilson_half_width_at_p050": wh,
            "inconclusive_at_N": bool(inconclusive),
        }
    )

paired_df = pd.DataFrame(paired_results)
print()
print("# Stats-rigor receipts:")
print(f"#   bootstrap_ci - n_resamples={N_RESAMPLES:,}, random_state={RANDOM_STATE}")
print("#   paired_delta_bootstrap - same settings")
print("#   paired_wilcoxon - two-sided, zero_method='wilcox' (drops zero-diff pairs)")
print("#   cohens_h - 2*arcsin(sqrt(p1)) - 2*arcsin(sqrt(p2))")
print("#   inconclusive_at_N flag - |delta| < 2 * Wilson half-width at p=0.5, N=n_pairs")
paired_df

In [ ]:
# Cell 6 - Failure-mode preview (placeholder)
# Real labels will live in docs/FAILURE_TAXONOMY.md as a CSV (see the
# template). This cell renders the horizontal stacked bar chart with
# synthetic mode counts so the plotting pipeline is proven before the
# Day 7 labeling pass produces real labels.
#
# When real labels exist, replace `_synthetic_failure_labels` with
# `pd.read_csv(REPO_ROOT / "docs" / "failure_labels.csv")`.

MODE_LABELS = (
    "trajectory_overshoot",
    "gripper_slip",
    "timeout",
    "wrong_object",
    "premature_release",
    "drift",
)


def _synthetic_failure_labels(
    df_: pd.DataFrame, n_per_cell: int = 5, seed: int = 1
) -> pd.DataFrame:
    """Sample failed episodes per (policy, env) cell, assign random modes."""
    rng = np.random.default_rng(seed)
    rows = []
    for _key, cell in df_[~df_["success"]].groupby(["policy", "env"]):
        if len(cell) == 0:
            continue
        sample = cell.sample(
            n=min(n_per_cell, len(cell)), random_state=int(rng.integers(0, 10_000))
        )
        for _, ep in sample.iterrows():
            rows.append(
                {
                    "policy": ep["policy"],
                    "env": ep["env"],
                    "seed": ep["seed"],
                    "episode_index": ep["episode_index"],
                    "mode": str(rng.choice(MODE_LABELS)),
                }
            )
    return pd.DataFrame(rows)


labels = _synthetic_failure_labels(df, n_per_cell=8, seed=1)

# Aggregate to (policy, env, mode) counts.
mode_counts = (
    labels.groupby(["policy", "env", "mode"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=MODE_LABELS, fill_value=0)
)

# Stacked horizontal bars - one row per (policy, env), bars segmented by mode.
fig, ax = plt.subplots(figsize=(8, 0.35 * len(mode_counts) + 1.5))
bottoms = np.zeros(len(mode_counts))
colors = plt.get_cmap("tab10")(np.arange(len(MODE_LABELS)))
for i, mode in enumerate(MODE_LABELS):
    counts = mode_counts[mode].to_numpy()
    ax.barh(
        np.arange(len(mode_counts)),
        counts,
        left=bottoms,
        color=colors[i],
        label=mode,
        edgecolor="white",
        linewidth=0.4,
    )
    bottoms += counts

labels_y = [f"{p} | {e}" for p, e in mode_counts.index]
ax.set_yticks(np.arange(len(mode_counts)))
ax.set_yticklabels(labels_y, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("# labeled failures")
ax.set_title("Failure mode breakdown - PLACEHOLDER (synthetic random labels)")
ax.legend(loc="upper right", fontsize=8, ncol=2, frameon=False)
fig.tight_layout()
plt.show()
mode_counts

## Cell 7 - The finding paragraph (DRAFT)

> **Status:** synthetic data. The numbers below are placeholders and the
> verbal claim is hedged accordingly. Replace this entire block on Day 7
> once the real sweep parquet is at `results/sweep.parquet`.

On synthetic data with the random baseline at $p = 0.04$ on PushT
($n = 250$), the bootstrap 95% CI on the random baseline's success rate
is approximately $[0.02, 0.07]$. The DiffusionPolicy / ACT pair is the
two-policy comparison that shares the most envs; on this synthetic
parquet DiffusionPolicy produces a $\Delta\text{success}$ around $+0.12$
on PushT vs ACT, which sits at the boundary of the MDE band given the
Wilson half-width at $p=0.5$, $N=250$ ($\approx 0.062$, so $2 \times \text{HW} \approx 0.124$);
the synthetic delta is therefore on the edge of being inconclusive at
this N — a useful sanity check that the pipeline does not over-claim.

When real data lands, this paragraph becomes one defensible non-obvious
sentence anchored on the largest cross-cell delta whose 95% CI excludes
zero AND whose Cohen's $h$ is at least medium ($|h| \geq 0.5$). If no
such delta exists in the real sweep, the abstract states that
explicitly per the writeup contract in CEO-PLAN § Headline finding:
"5 seeds × 50 episodes did not support a defensible cross-policy
finding at this N — see leaderboard."


In [ ]:
# Cell 8 - Reproducibility footer
# Print every coordinate a reader needs to re-derive the numbers above.
# Mirrors what the paper's footer prints into the Methods appendix.


def _git_sha(root: Path) -> str:
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=root,
            capture_output=True,
            text=True,
            check=False,
            timeout=5,
        )
        return result.stdout.strip() or "unknown"
    except (OSError, subprocess.TimeoutExpired):
        return "unknown"


print("Reproducibility footer")
print("=" * 50)
print(f"lerobot_bench.__version__  : {lerobot_bench.__version__}")
print(f"git rev-parse HEAD         : {_git_sha(REPO_ROOT)}")
print(f"results parquet path       : {RESULTS_PARQUET}")
print(f"using synthetic fallback   : {USING_SYNTHETIC}")
print(f"total rows                 : {len(df):,}")
print(f"unique (policy, env, seed) : {df.groupby(['policy', 'env', 'seed']).ngroups}")
print(f"n_resamples (bootstrap)    : {N_RESAMPLES:,}")
print(f"random_state (bootstrap)   : {RANDOM_STATE}")
print(f"CI level                   : {CI_LEVEL}")
print(f"python                     : {sys.version.split()[0]}")
print(
    f"numpy / pandas / matplotlib: {np.__version__} / {pd.__version__} / {plt.matplotlib.__version__}"
)